# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    load_commutative_cnn_pretraining_config,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state_v7.pt")
validation_fraction = 0.15
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    num_prototypes=64,
    probe_local_count=32,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.15,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=70,
    learning_rate=2e-4,
    weight_decay=3e-4,
    early_stopping_patience=16,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=30,
    early_stopping_monitor="loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=5,
    training_plot_dir="artifacts/pretrained_commutative_cnn/loss_plots",
    training_plot_every_n_epochs=2,
    training_plot_smoothing_window=5,
    scheduler_patience=5,
    scheduler_factor=0.75,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.10,
    cross_warmup_epochs=12,
    cross_ramp_epochs=32,
    prototype_temperature=0.20,
    prototype_alignment_weight=0.05,
    prototype_warmup_epochs=12,
    prototype_ramp_epochs=32,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
    probe_mask_probability=0.75,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=1.0,
    probe_alpha_frequency=1.0,
    probe_alpha_correlation=0.25,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Commutative CNN pretraining config in {pretraining_config_path}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Pretrained encoder checkpoint target: {pretrained_encoder_path.resolve()}")
pretraining_config


Commutative CNN pretraining config in /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml
Pretraining loss PDFs: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/loss_plots
Pretrained encoder checkpoint target: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/encoder_state_v7.pt


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v7.pt'), validation_fraction=0.15, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(16, 32), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(48, 64), temporal_st_kernel_sizes=(5, 3), temporal_ts_channels=(32, 48, 64), temporal_ts_kernel_sizes=(7, 5, 3), spatial_agg_channels=(32, 64), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1), spatial_agg_pool_kernel_z=(1, 1), spatial_agg_pool_kernel_xy=(1, 2), spatial_agg_pool_strid

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


{'all_tensors': torch.Size([2144, 20, 5, 96, 96]),
 'all_metadata': (2144, 7),
 'train_base_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'train_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'val_tensors': torch.Size([322, 20, 5, 96, 96]),
 'train_base_metadata': (1822, 7),
 'train_metadata': (1822, 7),
 'val_metadata': (322, 7)}

## Output Review

The interrupted run with `learning_rate=4e-4` improved validation loss at epoch 2 (`0.5846`) but then became unstable by epoch 4 (`0.9772`) while training loss kept falling. The saved `v5` loss curves show that self probes learn, but validation is dominated by noisy cross-probe and correlation terms after the cross-weight ramp starts. The completed `v6` run with `learning_rate=2e-4`, `lambda_cross=0.15`, warmup `16`, ramp `28`, and `lambda_align=0.02` selected `best_epoch=042` with smoothed `best_metric=5.3314`, but validation stayed volatile after the cross objective reached full strength. The next run keeps the optimizer settings, removes exact latent MSE alignment (`latent_alignment_weight=0.0`), lowers cross pressure to `lambda_cross=0.10`, and adds conservative prototype alignment (`num_prototypes=64`, `prototype_temperature=0.20`, `prototype_alignment_weight=0.05`, warmup `12`, ramp `32`). This keeps self-probes dominant early, lets teacher-student cross-probes and prototype alignment ramp together, and writes to `encoder_state_v7.pt` so the existing `encoder_state_v6.pt` checkpoint remains available.

In [ ]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


Commutative CNN model: parameters=174,283 trainable=174,283 size=0.67 MB
cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trS=train_self_probe_loss
    trX=train_cross_probe_loss
    trP=train_prototype_alignment_loss
    trLA=train_latent_alignment_loss
     ep       lr       eta |      trL      trS      trX      trP     trLA |      vaL      vaS      vaX      vaP     vaLA
001/070 2.00e-04  15:33:11 |   3.7970   3.7970   0.2583   4.5745   0.4556 |  29.2867  29.2867  36.3039   4.2925  62.0573
002/070 2.00e-04  15:01:49 |   2.9619   2.9619   0.3554   4.2819   0.4867 |  44.8290  44.8290  63.1462   4.5613  70.9051
003/070 2.00e-04  14:50:22 |   2.9359   2.9359   0.3766   4.1819   0.4316 |  14.7932  14.7932  24.5971   4.1941  14.6093
004/070 2.00e-04  14:40:57 |   2.8870   2.8870   0.4129   4.1624   0.3702 |  29.3848  29.3848  48.9147   4.1770  21.5988


In [ ]:
model.pretrain_history_.tail()